1. When should you start a new session with an injected structured summary rather than resuming a prior session with `--resume`?

A.Always — new sessions are always more reliable than resumed ones
**B.When the prior session's tool results are stale due to significant system state changes**
C.Only when the original session exceeded 100 turns
D.When the task domain changes from one programming language to another


Scenario: Multi-Agent Research System
Your coordinator agent delegates a broad research topic to four specialist subagents. The synthesis subagent's output has poor source attribution — it cannot identify which findings came from which source. What caused this and what is the fix?
<!-- A.The synthesis subagent needs a larger context window to hold all sources -->
B.The coordinator passed findings as plain concatenated text without preserving source metadata; fix by passing structured data that separates content from source URLs, titles, and page numbers
C.The web search subagent is not returning URLs in its results
D.Attribution requires a dedicated attribution subagent in the pipeline


A coordinator agent decomposes a research task into 3 fixed subtopics and consistently misses entire subject areas. What is the root cause?
A.The subagents are not returning their completed results back to the coordinator agent for aggregation
B.The context window is too small to handle all aspects of the full topic
<!-- C.Fixed decomposition is too narrow — the coordinator should adapt scope dynamically -->
D.The web search tool is returning incomplete or low-quality search results

During testing, combined outputs from the web search agent (85K tokens including page content) and the document analysis agent (70K tokens including reasoning chains) total 155K tokens, but the synthesis agent performs optimally with inputs under 50K tokens. What's the most effective solution?
A.Store findings in a vector database and give the synthesis agent retrieval tools to query during its work
B.Add an intermediate summarization agent that condenses findings before passing to synthesis
<!-- C.Modify upstream agents to return structured data (key facts, citations, relevance scores) instead of verbose content and reasoning -->
D.Have the synthesis agent process findings in sequential batches, maintaining running state between calls
✓ Correct — Explanation
Modifying upstream agents to return structured data addresses the root cause: the agents are returning far more token-volume than the synthesis step needs. Full page content and reasoning chains are intermediate artifacts of the search and analysis process — they are not the output the synthesis agent needs. Requiring agents to output key facts, citations, and relevance scores reduces token volume at the source while preserving the essential information. This is better than downstream approaches: vector retrieval (A) adds latency and requires the synthesis agent to know what to retrieve; intermediate summarization (B) adds another agent hop and may lose structured facts; sequential batches (D) require maintaining state across calls and do not reduce total token usage.


After a multi-agent research pipeline produces a comprehensive but shallow first draft, the coordinator needs to run iterative refinement. What is the architecturally correct approach?
A.Have the coordinator re-run the entire pipeline from scratch
B.Have the synthesis subagent expand all sections autonomously without coordinator guidance
<!-- C.The coordinator evaluates the synthesis for coverage gaps, re-delegates targeted queries to search/analysis subagents, and re-invokes synthesis until quality criteria are met -->
D.Ask the end user to identify which sections need more depth
✓ Correct — Explanation
Iterative refinement is a coordinator responsibility. The coordinator evaluates synthesis output against quality criteria (depth, coverage, citation density), identifies specific gaps (not vague dissatisfaction), formulates targeted follow-up queries, and re-delegates only what is needed. This loop continues until the coordinator's quality evaluation passes. Full pipeline re-runs are wasteful; user-driven gap identification is a design failure.


A subagent fails to retrieve data from a slow external API (timeout). What is the correct error-handling architecture?
A.Return an empty result set and let the coordinator decide what to do
B.Immediately propagate the error to the coordinator without any local recovery attempt
C.Attempt local recovery (e.g., retry); if unresolvable, propagate to coordinator with partial results and a structured error description
D.Log the error silently and continue as if the tool call succeeded
✓ Correct — Explanation
Subagents should attempt local recovery for transient errors (timeouts, temporary unavailability) before escalating. When escalating, they must return structured context: what data was retrieved before failure, what failed and why, and whether retry is appropriate. Silent error suppression prevents coordinator recovery; immediate escalation without retry wastes resources on recoverable failures.


You are designing a coordinator prompt for a multi-agent research system. Which prompt style leads to better subagent outcomes?
A.Step-by-step procedural instructions: "First search for X, then analyze Y, then synthesize Z"
B.Goal-oriented prompts specifying research goals and quality criteria rather than procedural steps
C.Single-sentence prompts to minimize token usage
D.Prompts that list which tools each subagent should use in what order
✗ Incorrect — Explanation
Goal-oriented coordinator prompts ("Research the impact of X on Y, ensure findings are cited, cover both short-term and long-term effects") outperform procedural step-by-step instructions. Specifying the *what* and *quality bar* rather than the *how* allows subagents to adapt their approach based on what they discover, producing more thorough and accurate results.


In an agentic loop, what does a `stop_reason` value of `"tool_use"` indicate?
A.Claude is requesting that a tool be executed and its result fed back
B.Claude has finished the task and no further action is needed
C.The agentic loop has exceeded its configured maximum iteration count limit and was terminated
D.Claude encountered an unrecoverable error and cannot continue
✗ Incorrect — Explanation
`stop_reason: "tool_use"` means Claude wants to invoke one or more tools. Your loop must execute those tools, append the results to the conversation history, and call Claude again. The loop only terminates when `stop_reason` is `"end_turn"` (or a budget/turn limit is hit).

An engineer asks your developer productivity agent to "add comprehensive tests to this legacy codebase." The agent immediately starts writing tests for the first file it finds, without understanding the codebase structure. What task decomposition pattern should the agent use?
A.Prompt chaining: write tests for each file in alphabetical order
B.Dynamic adaptive decomposition: first map codebase structure, identify high-impact areas, then create a prioritized plan that adapts as dependencies are discovered
C.Parallel decomposition: spawn one subagent per file simultaneously
D.Sequential fixed pipeline: analyze → write → run → fix
✓ Correct — Explanation
Open-ended tasks on unknown systems require dynamic adaptive decomposition. The agent cannot know the right approach until it understands the codebase. Phase 1: map the structure and understand the domain. Phase 2: identify high-impact, under-tested areas. Phase 3: create a prioritized plan that adapts as dependencies and test complexity emerge. This contrasts with prompt chaining, which requires knowing the steps upfront.


An interception hook blocks a `process_refund` call when the refund amount exceeds $500 and redirects to `escalate_to_human`. A colleague argues this logic should be in the system prompt instead. What is the strongest argument against the system prompt approach?
A.System prompts cannot reference specific dollar thresholds
B.Claude would ignore financial rules in system prompts
C.Hooks provide deterministic, guaranteed enforcement; prompt instructions are probabilistic and have a measurable failure rate on compliance-critical paths
D.System prompts are processed before tool definitions and cannot reference tool names
✓ Correct — Explanation
For business rule enforcement (refund thresholds, compliance gates), determinism is required. Hooks are code that executes deterministically — the rule fires every time without exception. A system prompt instruction might work 99% of the time, but for financial operations, 1% failure means real financial exposure. This is the fundamental programmatic vs prompt-based enforcement tradeoff.



`fork_session` is most appropriate when you want to:
A.Create independent exploration branches from a shared analysis baseline point
B.Continue a previously suspended session with updated and corrected context information
C.Run the same agent on multiple machines simultaneously for load balancing
D.Share session state between two different coordinator agents in a pipeline
✗ Incorrect — Explanation
`fork_session` creates parallel branches from a shared starting point — ideal when you want to explore two divergent approaches (e.g., two refactoring strategies, two testing architectures) without them interfering with each other. It is not for resuming sessions or distributed execution.



A customer support agent processes a refund request. The `process_refund` tool is called before `get_customer` has been invoked. Which root cause and fix apply?
A.Claude misunderstood the system prompt order; rewrite it with numbered steps
B.Missing programmatic prerequisite gate — `process_refund` should require a verified customer ID from a prior `get_customer` call in the current session to execute
C.The tools should be merged into a single `process_refund_with_lookup` tool
D.The agent should ask the user to confirm the customer ID before proceeding
✓ Correct — Explanation
This is the canonical prerequisite enforcement problem. The gate must be programmatic: before `process_refund` executes, the hook checks whether a verified customer ID exists in session state from a completed `get_customer` call. If not, the hook blocks the call and redirects to identity verification. Prompt-based ordering is insufficient for financial operations.



You use `--resume <session-name>` to continue a prior investigation. After resuming, you notice Claude is reasoning from stale file analysis done before you modified several files. What should you do?
A.Restart the entire session from scratch with no prior context at all
B.Use `fork_session` to create an independent branch from the stale point
C.Simply re-run the previous prompt — Claude will automatically detect and process file changes
D.Tell the resumed session which files changed so Claude re-analyzes them specifically
✗ Incorrect — Explanation
When resuming a session after code modifications, you must explicitly tell Claude which files changed. Claude will not automatically diff the file system against its prior analysis. Targeted re-analysis ("file X and Y were modified — please re-analyze them") is more efficient than full re-exploration and produces better results than relying on stale tool results.


A developer spent three hours analyzing a legacy codebase yesterday. Today they want to explore two different refactoring approaches starting from the same analysis baseline — without the approaches polluting each other. Which session management feature should they use?
A.--resume with yesterday's session name to continue from where they left off
B.Start two new sessions with the same system prompt and re-analyze each time
C./compact to reduce yesterday's session to a summary, then branch from there
D.fork_session to create two independent branches from the shared baseline
✗ Incorrect — Explanation
fork_session creates independent branches from an existing session's state — both branches share the analysis baseline but diverge independently from that point. Changes in branch A don't affect branch B. This is precisely the "explore two approaches from a common baseline" use case. --resume (A) would continue the single prior session, not create branches. Starting new sessions (C) requires expensive re-analysis. /compact reduces context but doesn't create forks.

A multi-agent research coordinator produces synthesis reports with poor coverage because it invokes all four subagents (web, document, synthesis, report) for every query regardless of complexity. What architectural improvement should be made?
A.Add a fifth "orchestration" subagent to manage the other four
B.Switch to a fixed sequential pipeline with no dynamic routing
C.Configure the coordinator to analyze query requirements and dynamically select only the relevant subagents
D.Increase the context window size to accommodate all four subagents' outputs
✓ Correct — Explanation
Always routing through the full pipeline is wasteful and can actually degrade quality (irrelevant subagents add noise). The coordinator should analyze each query and dynamically select only the subagents needed. A simple factual question might only need the web search subagent; a document-heavy analysis might skip web search entirely.

A developer implements an agentic loop that checks `response.content[0].text` to decide whether to stop. What is wrong with this approach?
A.The content array may be empty or missing, causing a runtime index error
B.Text content checks are unreliable — `stop_reason` is the authoritative signal
C.The developer should check `response.content[0].type` instead of the `.text` field
D.Nothing is wrong; this is a valid alternative approach to checking `stop_reason`
✓ Correct — Explanation
`stop_reason` is the correct and only reliable termination signal. Parsing text content (e.g., looking for "Done" or "Finished") is an anti-pattern: Claude's phrasing varies and the text may contain the word even mid-task. Always use `stop_reason === "end_turn"` to terminate the loop.


What is the role of a coordinator agent in a hub-and-spoke multi-agent architecture?
A.To execute all tool calls directly without delegating to others
B.To act as a proxy relay between subagents and external tool APIs
C.To decompose tasks, delegate to subagents, and aggregate results
D.To store a shared memory space accessible by all active subagents
✗ Incorrect — Explanation
In hub-and-spoke, the coordinator is the central hub. It decomposes the task, delegates to specialist subagents, routes information, and aggregates results. Subagents communicate only through the coordinator — never directly with each other.


When passing context from a web search subagent to a synthesis subagent, what should the coordinator include to preserve attribution?
A.Structured data separating content from metadata like source URLs and page numbers
B.Only the final text summary without source URLs; including metadata adds unnecessary token overhead cost
C.A plain text blob of all findings concatenated together without structure
D.A reference to the shared session ID where the subagent results are stored
✗ Incorrect — Explanation
The coordinator should use structured data to separate content (what was found) from metadata (where it came from — URLs, titles, page numbers). This allows the synthesis subagent to produce properly cited output and enables the coordinator to trace any finding back to its source for verification.

Scenario: You are building a customer support resolution agent using the Claude Agent SDK. The agent handles high-ambiguity requests like returns, billing disputes, and account issues. It has access to your backend systems through custom MCP tools (get_customer, lookup_order, process_refund, escalate_to_human). Your target is 80%+ first-contact resolution while knowing when to escalate.
Production data shows that in 12% of cases, your agent skips get_customer entirely and calls lookup_order using only the customer's stated name, occasionally leading to misidentified accounts and incorrect refunds. What change would most effectively address this reliability issue?
A.Add a programmatic prerequisite that blocks lookup_order and process_refund calls until get_customer has returned a verified customer ID
B.Enhance the system prompt to state that customer verification via get_customer is mandatory before any order operations
C.Add few-shot examples showing the agent always calling get_customer first, even when customers volunteer order details
D.Implement a routing classifier that analyzes each request and enables only the subset of tools appropriate for that request type
✓ Correct — Explanation
When a specific tool sequence is required for critical business logic (like verifying customer identity before processing refunds), programmatic enforcement provides deterministic guarantees that prompt-based approaches cannot. Options B and C rely on probabilistic LLM compliance, which is insufficient when errors have financial consequences. Option D addresses tool availability rather than tool ordering, which is not the actual problem.



